In [11]:
pip install fastapi uvicorn

In [6]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

intents['greetings']['patterns'].extend(['hello there', 'hi everyone', 'howdy', 'good morning', 'good afternoon'])
intents['goodbye']['patterns'].extend(['see ya', 'farewell', 'until next time', 'bye bye'])
intents['thanks']['patterns'].extend(['much appreciated', 'thanks a lot', 'thank you very much'])
intents['joke']['patterns'].extend(['another joke', 'make me laugh', 'tell me something funny'])
intents['time']['patterns'].extend(['current time', 'what is the time', 'do you know the time'])
intents['date']['patterns'].extend(['today\'s date', 'what day is it', 'what is today\'s date'])

# Add new intents
intents['name_query'] = {
    'patterns': ['what is your name', 'who are you', 'your name'],
    'responses': ['I am a chatbot!', 'You can call me Bot.', 'I don\'t have a name.']
}
intents['abilities_query'] = {
    'patterns': ['what can you do', 'your capabilities', 'help me'],
    'responses': ['I can answer questions, tell jokes, and provide the current time and date!', 'I can assist with various queries you might have.']
}
intents['weather_query'] = {
    'patterns': ['what\'s the weather like', 'weather today', 'forecast'],
    'responses': ['I cannot provide real-time weather information.', 'I am not equipped to check the weather.']
}

# Re-prepare data for intent detection with updated intents
train_data_updated = []
for intent, data in intents.items():
    for pattern in data['patterns']:
        train_data_updated.append((pattern, intent))

# Split data into input and output
X_updated = [x[0] for x in train_data_updated]
y_updated = [x[1] for x in train_data_updated]

# Re-initialize and re-vectorize input data to capture new vocabulary from added patterns
vectorizer_re_fit = TfidfVectorizer()
X_vectorized_updated = vectorizer_re_fit.fit_transform(X_updated)

# Perform Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # Changed n_splits from 5 to 3 to handle small classes
accuracies = []

for train_index, test_index in skf.split(X_vectorized_updated, y_updated):
    X_train_fold, X_test_fold = X_vectorized_updated[train_index], X_vectorized_updated[test_index] # Corrected variable name
    y_train_fold, y_test_fold = np.array(y_updated)[train_index], np.array(y_updated)[test_index]

    clf_fold = MultinomialNB()
    clf_fold.fit(X_train_fold, y_train_fold)
    y_pred_fold = clf_fold.predict(X_test_fold)
    accuracies.append(accuracy_score(y_test_fold, y_pred_fold))

mean_accuracy = np.mean(accuracies);
print(f'Mean Intent detection accuracy using Stratified K-Fold: {mean_accuracy}')

# For further use in the chatbot, you would generally retrain the model
# on the entire dataset after validating performance with cross-validation.
# Here we will keep the last trained classifier (clf_fold) and vectorizer_re_fit
# for potential subsequent steps, but note that `clf_fold` is from the last fold.

Mean Intent detection accuracy using Stratified K-Fold: 0.897089947089947


In [4]:
# Retrain the final model on the entire updated dataset
vectorizer = vectorizer_re_fit # Use the vectorizer fitted on the full updated dataset
clf = MultinomialNB() # Initialize a new classifier
clf.fit(X_vectorized_updated, y_updated) # Train on the entire updated and vectorized data

print("Chatbot model successfully retrained with improved patterns. You can now re-run the chatbot function to test it.")

Chatbot model successfully retrained with improved patterns. You can now re-run the chatbot function to test it.


In [7]:
import pickle

# Define filenames for the model and vectorizer
model_filename = 'chatbot_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

# Save the trained classifier (clf) to a file
with open(model_filename, 'wb') as file:
    pickle.dump(clf, file)

# Save the trained vectorizer (vectorizer) to a file
with open(vectorizer_filename, 'wb') as file:
    pickle.dump(vectorizer, file)

print(f"Model saved to {model_filename}")
print(f"Vectorizer saved to {vectorizer_filename}")

Model saved to chatbot_model.pkl
Vectorizer saved to tfidf_vectorizer.pkl


You can load the saved model and vectorizer back into your application like this:

In [8]:
# Load the model and vectorizer back from files
with open('chatbot_model.pkl', 'rb') as file:
    loaded_clf = pickle.load(file)

with open('tfidf_vectorizer.pkl', 'rb') as file:
    loaded_vectorizer = pickle.load(file)

print("Model and vectorizer loaded successfully!")

# You can now use loaded_clf and loaded_vectorizer for predictions
# For example, to process an input with the loaded model:
# user_input_test = "hello"
# vector = loaded_vectorizer.transform([user_input_test.lower()])
# intent = loaded_clf.predict(vector)[0]
# print(f"Detected intent for '{user_input_test}': {intent}")

Model and vectorizer loaded successfully!


In [9]:
dependencies = [
    'nltk',
    'scikit-learn',
    'numpy'
]

with open('requirements.txt', 'w') as f:
    for dep in dependencies:
        f.write(f"{dep}\n")

print("requirements.txt created successfully!")


requirements.txt created successfully!


The `requirements.txt` file has been created. You can view its content by running a shell command like `!cat requirements.txt` or by navigating to the file in the file browser.

In [10]:
pip install fastapi uvicorn

In [12]:
pip install -r requirements.txt

In [15]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import nltk
from nltk.stem import WordNetLemmatizer
import random
import datetime

# Download NLTK data if not already present (only needed once)
# You might want to pre-download these in your Dockerfile if deploying
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

app = FastAPI()

# Load the trained model and vectorizer
model_filename = 'chatbot_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

try:
    with open(model_filename, 'rb') as file:
        clf = pickle.load(file)
    with open(vectorizer_filename, 'rb') as file:
        vectorizer = pickle.load(file)
    print("Model and vectorizer loaded successfully!")
except FileNotFoundError:
    print(f"Error: Model or vectorizer file not found. Make sure '{model_filename}' and '{vectorizer_filename}' are in the same directory.")
    # You might want to exit or handle this more gracefully in a real app
    exit()

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Re-define your intents dictionary (ensure it matches what your model was trained on)
intents = {
    'greetings': {
        'patterns': ['hello', 'hi', 'hey', 'hello there', 'hi everyone', 'howdy', 'good morning', 'good afternoon'],
        'responses': ['Hello!', 'Hi there!', 'Hey, how can I help?']
    },
    'goodbye': {
        'patterns': ['bye', 'goodbye', 'see you', 'see ya', 'farewell', 'until next time', 'bye bye'],
        'responses': ['See you later!', 'Goodbye!', 'Have a great day!']
    },
    'thanks': {
        'patterns': ['thank you', 'thanks', 'much appreciated', 'thanks a lot', 'thank you very much'],
        'responses': ['You\'re welcome!', 'No problem!', 'Glad I could help!']
    },
    'joke': {
        'patterns': ['joke', 'tell me a joke', 'another joke', 'make me laugh', 'tell me something funny'],
        'responses': [
            'Why don\'t scientists trust atoms? Because they make up everything!',
            'Why don\'t eggs tell jokes? They\'ll crack each other up!',
            'Why did the scarecrow win an award? Because he was outstanding in his field!'
        ]
    },
    'time': {
        'patterns': ['time', 'what time is it', 'current time', 'what is the time', 'do you know the time'],
        'responses': [] # Responses are dynamically generated
    },
    'date': {
        'patterns': ['date', 'what\'s the date', 'today\'s date', 'what day is it', 'what is today\'s date'],
        'responses': [] # Responses are dynamically generated
    },
    'name_query': {
        'patterns': ['what is your name', 'who are you', 'your name'],
        'responses': ['I am a chatbot!', 'You can call me Bot.', 'I don\'t have a name.']
    },
    'abilities_query': {
        'patterns': ['what can you do', 'your capabilities', 'help me'],
        'responses': ['I can answer questions, tell jokes, and provide the current time and date!', 'I can assist with various queries you might have.']
    },
    'weather_query': {
        'patterns': ['what\'s the weather like', 'weather today', 'forecast'],
        'responses': ['I cannot provide real-time weather information.', 'I am not equipped to check the weather.']
    }
}

# Define the request body model
class ChatRequest(BaseModel):
    message: str

# The chatbot processing function (adapted from your notebook)
def process_input(user_input):
    # Tokenize the input
    tokens = nltk.word_tokenize(user_input.lower())

    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # Detect intent
    vector = vectorizer.transform([' '.join(tokens)])
    intent = clf.predict(vector)[0]

    # Recognize entities (simplified for this example)
    entities = []
    for token, pos in nltk.pos_tag(tokens):
        if pos == 'NNP':  # Proper Noun
            entities.append(token)

    # Generate response
    if intent == 'time':
        return f'Current time is {datetime.datetime.now().strftime("%H:%M:%S")}'
    elif intent == 'date':
        return f'Today\'s date is {datetime.datetime.now().strftime("%Y-%m-%d")}'
    else:
        if intent in intents and intents[intent]['responses']:
            response = random.choice(intents[intent]['responses'])
            if entities:
                # Simple entity integration, adjust as needed
                response += f' {entities[0]}'
            return response
        else:
            return "I'm not sure how to respond to that."

# Define your API endpoint
@app.post("/chat")
async def chat(request: ChatRequest):
    user_message = request.message
    chatbot_response = process_input(user_message)
    return {"response": chatbot_response}

# Optional: Add a root endpoint for health check
@app.get("/ मिळू")
async def root():
    return {"message": "Chatbot API is running!"}


Model and vectorizer loaded successfully!


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
get_ipython().system('uvicorn main:app --reload')

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [13261] using WatchFiles
ERROR:    Error loading ASGI app. Could not import module "main".
